# Steam game lifespan preprocessing

This notebook prepares Steam player-count histories for survival analysis. The main output is a **game-day panel**: one row per app and calendar day, with player-count summaries, observation age, and an explicit indicator for whether the game appears to have reached the end of the observation window.

## What to focus on

- Preserve the original five-minute observations for auditability, but aggregate to daily values for a tractable time-varying dataset.
- Treat the end of the available history as right-censoring. A low player count is not automatically a death event.
- Avoid look-ahead bias: compute covariates from observations on or before each day.
- Keep missingness and data-quality flags; do not silently turn missing history into zero players.
- Join release metadata so lifespan can be measured from release date, while retaining games with incomplete metadata for review.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "data-preprocessing":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

DATA_ROOT = PROJECT_ROOT / "data" / "steam-games-dataset"
HISTORY_DIRS = [
    DATA_ROOT / "PlayerCountHistoryPart1",
    DATA_ROOT / "PlayerCountHistoryPart2",
]
METADATA_PATH = DATA_ROOT / "applicationInformation.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

# A sustained low-count rule is a candidate event definition, not a fact about game closure.
LOW_PLAYER_THRESHOLD = 1
LOW_PLAYER_DAYS_REQUIRED = 28
MIN_HISTORY_DAYS = 7

assert DATA_ROOT.exists(), f"Dataset directory not found: {DATA_ROOT}"
assert METADATA_PATH.exists(), f"Application metadata not found: {METADATA_PATH}"
HISTORY_DIRS = [path for path in HISTORY_DIRS if path.exists()]
assert HISTORY_DIRS, "No player-count history directories were found."

print(f"Project root: {PROJECT_ROOT}")
print(f"History directories: {len(HISTORY_DIRS)}")

Project root: C:\Users\kanci\OneDrive\Dokumenty\GitHub\steam-games-life-analysis
History directories: 2


## 1. Load and standardize five-minute histories

Each history file is keyed by its numeric filename (`appid.csv`). We retain the source partition and filename so conflicting or duplicated files can be investigated instead of silently overwritten.

In [4]:
def appid_from_path(path: Path) -> int:
    match = re.fullmatch(r"(\d+)\.csv", path.name, flags=re.IGNORECASE)
    if match is None:
        raise ValueError(f"Unexpected history filename: {path}")
    return int(match.group(1))


def load_daily_histories(history_dirs: list[Path]) -> pd.DataFrame:
    daily_frames = []
    files_seen = 0
    for history_dir in history_dirs:
        for path in sorted(history_dir.glob("*.csv")):
            files_seen += 1
            frame = pd.read_csv(path, usecols=["Time", "Playercount"])
            frame["timestamp"] = pd.to_datetime(frame.pop("Time"), errors="coerce")
            frame["player_count"] = pd.to_numeric(frame.pop("Playercount"), errors="coerce")
            frame = frame.dropna(subset=["timestamp", "player_count"])
            frame = frame[frame["player_count"] >= 0].copy()
            frame["date"] = frame["timestamp"].dt.normalize()
            daily = (
                frame.groupby("date", as_index=False)
                .agg(
                    mean_player_count=("player_count", "mean"),
                    median_player_count=("player_count", "median"),
                    max_player_count=("player_count", "max"),
                    min_player_count=("player_count", "min"),
                    observation_count=("player_count", "size"),
                    first_timestamp=("timestamp", "min"),
                    last_timestamp=("timestamp", "max"),
                )
            )
            daily["appid"] = appid_from_path(path)
            daily["source_partition"] = history_dir.name
            daily_frames.append(daily)

    if not daily_frames:
        raise ValueError("No player-count CSV files were found.")
    result = pd.concat(daily_frames, ignore_index=True)
    print(f"Files processed: {files_seen:,}")
    return result.sort_values(["appid", "date"]).reset_index(drop=True)


daily_history = load_daily_histories(HISTORY_DIRS)
print(f"Game-days loaded: {len(daily_history):,}")
print(f"Games loaded: {daily_history['appid'].nunique():,}")
print(f"Date range: {daily_history['date'].min().date()} to {daily_history['date'].max().date()}")

Files processed: 2,000
Game-days loaded: 1,939,902
Games loaded: 1,998
Date range: 2017-12-14 to 2020-08-12


## 2. Join metadata and inspect coverage

Release dates are parsed with `dayfirst=True` because the source uses values such as `21-Dec-17`. Metadata is a left join: player histories remain in the panel even when a release date or title is missing.

In [6]:
metadata = pd.read_csv(METADATA_PATH, low_memory=False, encoding="cp1252")
metadata["appid"] = pd.to_numeric(metadata["appid"], errors="coerce").astype("Int64")
metadata["releasedate"] = pd.to_datetime(metadata["releasedate"], errors="coerce", dayfirst=True)
metadata["freetoplay"] = pd.to_numeric(metadata["freetoplay"], errors="coerce").astype("Int8")
metadata = metadata[["appid", "type", "name", "releasedate", "freetoplay"]].drop_duplicates("appid")

panel = daily_history.merge(metadata, on="appid", how="left", validate="many_to_one")
panel["metadata_missing"] = panel["name"].isna()
panel["date"] = pd.to_datetime(panel["date"])
panel["days_since_release"] = (panel["date"] - panel["releasedate"]).dt.days
panel["before_release"] = panel["days_since_release"].lt(0)

coverage = (
    panel.groupby("appid", as_index=False)
    .agg(
        first_observed=("date", "min"),
        last_observed=("date", "max"),
        observed_days=("date", "nunique"),
        metadata_missing=("metadata_missing", "max"),
        release_date_missing=("releasedate", lambda values: values.isna().all()),
    )
)
coverage["calendar_span_days"] = (coverage["last_observed"] - coverage["first_observed"]).dt.days + 1
coverage["coverage_ratio"] = coverage["observed_days"] / coverage["calendar_span_days"]

print(f"Metadata rows: {len(metadata):,}")
print(f"History games without metadata name: {coverage['metadata_missing'].sum():,}")
print(f"Games without release date: {coverage['release_date_missing'].sum():,}")
print(coverage["coverage_ratio"].describe().round(3))

C:\Users\kanci\AppData\Local\Temp\ipykernel_13724\1640305205.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  metadata["releasedate"] = pd.to_datetime(metadata["releasedate"], errors="coerce", dayfirst=True)


Metadata rows: 2,000
History games without metadata name: 0
Games without release date: 144
count    1998.000
mean        0.999
std         0.006
min         0.744
25%         0.999
50%         1.000
75%         1.000
max         1.000
Name: coverage_ratio, dtype: float64


## 3. Create time-varying covariates and survival durations

Rolling features include the current day and earlier days only. A gap in the daily history resets the low-player streak, so missing observations are never interpreted as zero players. The endpoint is right-censored because the dataset ends on a collection date; the low-player run is retained as a diagnostic for a later, sensitivity-tested event definition.

In [7]:
panel = panel.sort_values(["appid", "date"]).reset_index(drop=True)
panel["log_mean_player_count"] = np.log1p(panel["mean_player_count"])
panel["log_max_player_count"] = np.log1p(panel["max_player_count"])
panel["daily_change_log_players"] = panel.groupby("appid")["log_mean_player_count"].diff()
panel["observation_gap_days"] = panel.groupby("appid")["date"].diff().dt.days
panel["is_low_player_day"] = panel["median_player_count"].le(LOW_PLAYER_THRESHOLD)

# A streak is valid only across consecutive calendar days.
panel["streak_group"] = (
    panel["observation_gap_days"].fillna(1).ne(1).groupby(panel["appid"]).cumsum()
)
panel["low_streak_days"] = (
    panel["is_low_player_day"].groupby([panel["appid"], panel["streak_group"]]).cumsum()
)
panel.loc[~panel["is_low_player_day"], "low_streak_days"] = 0

for window in (7, 28):
    panel[f"rolling_{window}d_mean_log_players"] = (
        panel.groupby("appid")["log_mean_player_count"]
        .transform(lambda values: values.rolling(window, min_periods=1).mean())
    )

panel["survival_start"] = panel["releasedate"].fillna(panel["first_timestamp"])
panel["duration_days"] = (panel["date"] - panel["survival_start"]).dt.days
panel["usable_for_release_analysis"] = panel["releasedate"].notna() & panel["duration_days"].ge(0)
panel["right_censored"] = True

# One row per game for Kaplan-Meier/Cox setup. No game is called dead here.
game_summary = (
    panel.groupby("appid", as_index=False)
    .agg(
        name=("name", "first"),
        type=("type", "first"),
        freetoplay=("freetoplay", "first"),
        releasedate=("releasedate", "first"),
        first_observed=("date", "min"),
        last_observed=("date", "max"),
        observed_days=("date", "nunique"),
        max_player_count=("max_player_count", "max"),
        max_low_streak_days=("low_streak_days", "max"),
        usable_for_release_analysis=("usable_for_release_analysis", "max"),
    )
)
game_summary["duration_days"] = (
    game_summary["last_observed"] - game_summary["releasedate"]
).dt.days
# The current dataset provides an observation window, not confirmed closures.
game_summary["event"] = 0
game_summary["right_censored"] = True

game_summary.head()

,appid,name,type,freetoplay,releasedate,first_observed,last_observed,observed_days,max_player_count,max_low_streak_days,usable_for_release_analysis,duration_days,event,right_censored
0,10,Counter-Strike,game,0,2000-11-01,2017-12-14,2020-08-12,973,31930.0,0,True,7224.0,0,True
1,20,Team Fortress Classic,game,0,1999-04-01,2017-12-14,2020-08-12,972,193.0,0,True,7804.0,0,True
2,30,Day of Defeat,game,0,2003-05-01,2017-12-14,2020-08-12,971,354.0,0,True,6313.0,0,True
3,50,Half-Life: Opposing Force,game,0,1999-11-01,2017-12-14,2020-08-12,972,616.0,0,True,7590.0,0,True
4,70,Half-Life,game,0,1998-11-08,2017-12-14,2020-08-12,973,6022.0,0,True,7948.0,0,True


## 4. Validate and export

The exported panel is suitable for time-varying survival models after the analyst supplies an event definition and, where needed, Twitch covariates. The summary table deliberately has `event = 0` for every game: with this dataset alone, the last observed day is an administrative censoring date, not proof of death.

In [8]:
assert panel[["appid", "date"]].duplicated().sum() == 0
assert panel["mean_player_count"].ge(0).all()
assert panel["observation_count"].ge(1).all()
assert panel.groupby("appid")["date"].apply(lambda values: values.is_monotonic_increasing).all()
assert game_summary["event"].eq(0).all()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
panel_path = OUTPUT_DIR / "steam_game_daily_panel.csv"
summary_path = OUTPUT_DIR / "steam_game_survival_summary.csv"
panel.to_csv(panel_path, index=False)
game_summary.to_csv(summary_path, index=False)

print(f"Wrote panel: {panel_path} ({len(panel):,} rows)")
print(f"Wrote summary: {summary_path} ({len(game_summary):,} rows)")
print("Next modeling step: merge Twitch observations by app/channel and date, then define and justify an observed failure event.")

Wrote panel: C:\Users\kanci\OneDrive\Dokumenty\GitHub\steam-games-life-analysis\data\processed\steam_game_daily_panel.csv (1,939,902 rows)
Wrote summary: C:\Users\kanci\OneDrive\Dokumenty\GitHub\steam-games-life-analysis\data\processed\steam_game_survival_summary.csv (1,998 rows)
Next modeling step: merge Twitch observations by app/channel and date, then define and justify an observed failure event.
